# Food-101 (20 classes): transfer learning with ResNet-50

**Owner:** Member 2 (name)

ResNet-50 pretrained on ImageNet, with a new 20-class output layer. Data split, training loop and metrics come from `common.py`, shared with the scratch CNN notebook.

**On Google Colab:** Runtime → Change runtime type → GPU, then run this cell. Skip it when running locally.

In [ ]:
# !git clone https://github.com/Tinle0301/food101-cecs456.git
# %cd food101-cecs456
# !pip install -q -r requirements.txt

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from common import *

set_seed()
print('Device:', get_device())

## 1. Data
Same split as the scratch CNN (`split.json`): 600 train / 150 validation / 250 test images per class.

In [ ]:
# Set QUICK_TEST = True to check everything runs on a small subset first
QUICK_TEST = False
LIMIT = 300 if QUICK_TEST else 0

In [ ]:
IMG_SIZE, BATCH_SIZE = 224, 32
loaders = get_loaders(IMG_SIZE, BATCH_SIZE, limit=LIMIT)
print({k: len(v.dataset) for k, v in loaders.items()})

## 2. Model
ResNet-50 with ImageNet weights. The final layer (2048 → 1000) is replaced by dropout + a linear layer (2048 → 20).

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, len(CLASSES)))


def set_trainable(stage):
    """'head': train only the new final layer. 'finetune': also train the last residual block (layer4)."""
    for p in model.parameters():
        p.requires_grad = False
    for module in ([model.fc] if stage == 'head' else [model.fc, model.layer4]):
        for p in module.parameters():
            p.requires_grad = True
    print(f'{stage}: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters')


set_trainable('head')

## 3. Training
- Epochs 1–3: backbone frozen, train only the new head (lr 1e-3).
- Epochs 4–10: unfreeze `layer4` and fine-tune with a 10× smaller learning rate (lr 1e-4).

AdamW with weight decay 1e-4 and a cosine schedule. The weights from the epoch with the best validation macro-F1 are kept.

In [ ]:
EPOCHS = 2 if QUICK_TEST else 10
FREEZE_EPOCHS = 1 if QUICK_TEST else 3


def make_optimizer(lr, epochs):
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    return opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)


def unfreeze_after_head_training(epoch):
    if epoch == FREEZE_EPOCHS + 1:
        set_trainable('finetune')
        return make_optimizer(1e-4, EPOCHS - FREEZE_EPOCHS)


optimizer, scheduler = make_optimizer(1e-3, FREEZE_EPOCHS)
hist = train_model(model, loaders, optimizer, EPOCHS, 'transfer', scheduler,
                   on_epoch_start=unfreeze_after_head_training)

In [ ]:
plot_curves(hist, 'ResNet-50');

## 4. Test set results

In [ ]:
res = evaluate(model, loaders, 'transfer')

In [ ]:
plot_confusion(res, 'ResNet-50');

In [ ]:
plot_top_losses(res, loaders, 'ResNet-50');